# Análise do Consumo de Energia de um Data Center

**Projeto de Métodos Numéricos - 2º GQ - Grupo Eta**

Integrantes: Breno, Guilherme, Mirna, Manuela, Allan e Jeanne

> Este notebook importa as funções do pacote `projeto_2GQ.analise_energia` e as usa para resolver cada item.
> As implementações dos métodos ficam no pacote, não aqui.

## 1. O problema

A potência elétrica consumida por um data center foi medida a cada 2 horas, ao longo de 24 horas, totalizando 13 medições (de 0 h a 24 h). A energia é cobrada por faixa de horário, com uma tarifa (R$/kWh) diferente para cada um dos 12 intervalos de 2 horas.

| Hora (h) | 0 | 2 | 4 | 6 | 8 | 10 | 12 | 14 | 16 | 18 | 20 | 22 | 24 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Potência (kW) | 122,12 | 120,11 | 125,23 | 134,21 | 138,54 | 155,46 | 179,27 | 183,07 | 171,07 | 158,56 | 137,63 | 126,26 | 123,60 |

| Intervalo | 0–2 | 2–4 | 4–6 | 6–8 | 8–10 | 10–12 | 12–14 | 14–16 | 16–18 | 18–20 | 20–22 | 22–24 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Tarifa (R$/kWh) | 0,65 | 0,65 | 0,65 | 0,75 | 0,75 | 0,75 | 0,75 | 0,85 | 1,20 | 1,20 | 0,85 | 0,65 |

Pede-se:

- **(a)** a energia total consumida em 24 horas;
- **(b)** o custo total da energia consumida, considerando a tarifa de cada faixa;
- **(c)** o consumo das primeiras 12 h e das últimas 12 h, qual período consumiu mais e a diferença relativa, com o primeiro período como referência;
- **(d)** supondo uma redução de 10% no consumo em um único intervalo de 2 horas, qual intervalo gera a maior economia financeira e quanto ela representa em um mês de 30 dias.

### Metodologia

A energia é a integral da potência no tempo, $E = \int_0^{24} P(t)\,dt$. Como a potência só é conhecida nos instantes medidos, a integral é aproximada pela **regra dos trapézios**, aplicada a cada intervalo de 2 horas:

$$E_i = \frac{P_i + P_{i+1}}{2} \cdot h, \qquad h = 2\ \text{h}$$

A energia de cada intervalo é guardada separadamente, porque os itens (b), (c) e (d) dependem dela: o custo e a economia são calculados intervalo a intervalo, com a tarifa da faixa correspondente.

## 2. Leitura dos dados

*Responsável: Breno*

Ler os arquivos `consumo.csv` e `tarifas.csv` e organizar os dados que as outras etapas usam:
os horários das medições, as potências medidas e a tarifa de cada intervalo de 2 horas.

In [1]:
from projeto_2GQ.analise_energia.leitura import horas, potencias, tarifas

print("Hora (h) | Potência (kW)")
for i in range(len(horas)):
    print(f"{horas[i]:8d} | {potencias[i]:13.2f}")

print()
print("Intervalo | Tarifa (R$/kWh)")
for i in range(len(tarifas)):
    print(f"{horas[i]:02d}-{horas[i + 1]:02d}h    | {tarifas[i]:15.2f}")

Hora (h) | Potência (kW)
       0 |        122.12
       2 |        120.11
       4 |        125.23
       6 |        134.21
       8 |        138.54
      10 |        155.46
      12 |        179.27
      14 |        183.07
      16 |        171.07
      18 |        158.56
      20 |        137.63
      22 |        126.26
      24 |        123.60

Intervalo | Tarifa (R$/kWh)
00-02h    |            0.65
02-04h    |            0.65
04-06h    |            0.65
06-08h    |            0.75
08-10h    |            0.75
10-12h    |            0.75
12-14h    |            0.75
14-16h    |            0.85
16-18h    |            1.20
18-20h    |            1.20
20-22h    |            0.85
22-24h    |            0.65


## 3. Item (a) - Energia total consumida em 24 horas

*Responsável: Mirna*

Calcular a energia consumida em cada intervalo de 2 horas pela **regra dos trapézios** e somar tudo
para obter a energia total do dia, em kWh.

Justificar a escolha do método e discutir a precisão da aproximação.

In [2]:
from projeto_2GQ.analise_energia.trapezios import calcular_energias

energias, energia_total = calcular_energias(list(zip(horas, potencias)))

print("Intervalo | P inicial (kW) | P final (kW) | Energia (kWh)")
for i in range(len(energias)):
    print(f"{horas[i]:02d}-{horas[i + 1]:02d}h    | {potencias[i]:14.2f} | {potencias[i + 1]:12.2f} | {energias[i]:13.2f}")
print(f"Energia total: {energia_total:.2f} kWh")

Intervalo | P inicial (kW) | P final (kW) | Energia (kWh)
00-02h    |         122.12 |       120.11 |        242.23
02-04h    |         120.11 |       125.23 |        245.34
04-06h    |         125.23 |       134.21 |        259.44
06-08h    |         134.21 |       138.54 |        272.75
08-10h    |         138.54 |       155.46 |        294.00
10-12h    |         155.46 |       179.27 |        334.73
12-14h    |         179.27 |       183.07 |        362.34
14-16h    |         183.07 |       171.07 |        354.14
16-18h    |         171.07 |       158.56 |        329.63
18-20h    |         158.56 |       137.63 |        296.19
20-22h    |         137.63 |       126.26 |        263.89
22-24h    |         126.26 |       123.60 |        249.86
Energia total: 3504.54 kWh


## 4. Item (b) - Custo da energia consumida

*Responsável: Guilherme*

Multiplicar a energia de cada intervalo pela tarifa da faixa correspondente e somar para obter
o custo total, em R$. Montar uma tabela com intervalo, energia, tarifa e custo.

O cálculo deve ser feito intervalo a intervalo, nunca com uma tarifa média.

In [4]:
from projeto_2GQ.analise_energia.leitura import horas, potencias, tarifas
from projeto_2GQ.analise_energia.trapezios import calcular_energias
from projeto_2GQ.analise_energia.custo import calcular_custos

energias, energia_total = calcular_energias(list(zip(horas, potencias)))
custos, custo_total = calcular_custos(energias, tarifas)

print("Intervalo | Energia (kWh) | Tarifa (R$/kWh) | Custo (R$)")
for i in range(len(custos)):
    print(f"{horas[i]:02d}-{horas[i + 1]:02d}h    | {energias[i]:13.2f} | {tarifas[i]:15.2f} | {custos[i]:10.2f}")
print(f"Custo total: R$ {custo_total:.2f}")

Intervalo | Energia (kWh) | Tarifa (R$/kWh) | Custo (R$)
00-02h    |        242.23 |            0.65 |     157.45
02-04h    |        245.34 |            0.65 |     159.47
04-06h    |        259.44 |            0.65 |     168.64
06-08h    |        272.75 |            0.75 |     204.56
08-10h    |        294.00 |            0.75 |     220.50
10-12h    |        334.73 |            0.75 |     251.05
12-14h    |        362.34 |            0.75 |     271.75
14-16h    |        354.14 |            0.85 |     301.02
16-18h    |        329.63 |            1.20 |     395.56
18-20h    |        296.19 |            1.20 |     355.43
20-22h    |        263.89 |            0.85 |     224.31
22-24h    |        249.86 |            0.65 |     162.41
Custo total: R$ 2872.14


## 5. Item (c) - Comparação entre as primeiras 12 h e as últimas 12 h

*Responsável: Jeanne*

Calcular o consumo de energia das primeiras 12 horas e das últimas 12 horas, dizer qual período
consumiu mais e calcular a diferença relativa, usando o primeiro período como referência.

In [5]:
from projeto_2GQ.analise_energia.comparacao import comparar_periodos

e1, e2, maior, dif_real = comparar_periodos(energias)

print(f"Primeiras 12h: {e1:.2f} kWh")
print(f"Últimas 12h: {e2:.2f} kWh")
print(f"Período com maior consumo: {maior}")
print(f"Diferença relativa: {dif_real:.2f}%")

Primeiras 12h: 1648.49 kWh
Últimas 12h: 1856.05 kWh
Período com maior consumo: ultimas 12h
Diferença relativa: 12.59%


## 6. Item (d) - Redução de 10% em um intervalo de 2 h

*Responsável: Allan*

Supondo uma redução de 10% no consumo em um único intervalo de 2 horas, descobrir qual intervalo
gera a maior economia financeira e estimar essa economia em um mês de 30 dias.

## 7. Conclusões

*Responsável: Manuela*

### Resumo dos resultados

| Item | Resultado |
|---|---|
| (a) Energia total em 24 h | **3 504,54 kWh** |
| (b) Custo total do dia | **R$ 2 872,14** |
| (c) Primeiras 12 h × últimas 12 h | 1 648,49 kWh × 1 856,05 kWh — as **últimas 12 h** consumiram **12,59% a mais** |
| (d) Melhor intervalo para reduzir 10% | **16–18 h**: economia de R$ 39,56 por dia, cerca de **R$ 1 186,67 em 30 dias** |

O consumo cresce ao longo da manhã, atinge o pico entre 12 h e 16 h (acima de 180 kW) e cai à noite. Por isso a segunda metade do dia concentra mais energia.

O custo, porém, não acompanha só a energia. O intervalo de 12–14 h é o de maior consumo (362,34 kWh), mas o de maior custo é 16–18 h (R$ 395,56), porque ali a tarifa sobe para R$ 1,20/kWh. As duas faixas de R$ 1,20 (16–20 h) representam cerca de 26% do custo do dia com apenas 18% da energia. É por isso que, no item (d), a redução de 10% rende mais em 16–18 h: a economia depende do produto energia × tarifa, e não só do consumo.

### Justificativa do método

Como só conhecemos a potência a cada 2 horas, a energia (integral da potência no tempo) precisa ser aproximada numericamente. Escolhemos a **regra dos trapézios** porque:

- usa exatamente os pontos medidos, sem precisar de uma função que descreva a potência;
- considera a potência no início **e** no fim de cada intervalo. Somar só as potências equivale a usar retângulos, que supõem a potência constante durante as 2 horas;
- gera a energia de cada intervalo separadamente, o que é necessário para os itens (b), (c) e (d), já que cada faixa tem a sua tarifa.

Para comparação: retângulos pela esquerda dariam 3 503,06 kWh e pela direita 3 506,02 kWh. O trapézio (3 504,54 kWh) é a média dos dois.

### Precisão e limitações

- O trapézio supõe que a potência varia em linha reta entre duas medições. O erro de cada intervalo é proporcional a $h^3 \cdot f''(\xi)$, com $h = 2$ h; ele é maior onde a curva muda de direção, como perto do pico do meio-dia.
- Com medições a cada 2 horas, picos ou quedas curtos entre duas leituras não aparecem nos dados. Medições mais frequentes reduziriam esse erro.
- A diferença entre as aproximações por retângulos (cerca de 3 kWh em 3 500 kWh, menos de 0,1%) indica que, para este perfil de consumo, o erro do método é pequeno diante do total.
- A economia mensal do item (d) supõe que todos os 30 dias têm o mesmo perfil de consumo e as mesmas tarifas, e que a redução de 10% vale para o intervalo inteiro.
